# 02 — Cleaning & EDA (Week 2)

**Goal:** Turn raw drive-day CSVs into a labelled drive-level snapshot dataset
and explore class imbalances and SMART attribute distributions.

Everything that touches the full dataset runs through DuckDB SQL — nothing here loads
all drive-days into pandas. The only pandas DataFrame built is the final snapshot
table, which is one row per drive (thousands of rows, not tens of millions).


In [1]:
import sys
sys.path.append("../src")
import load_data as ld
import cleaning as cl

con = ld.get_connection()

# TODO: swap to "../data/raw" once your real Q1 2025 CSVs are unzipped there
DATA_DIR = "../data/raw"


## 1. Recover Data Exploration's Target Framing

In [2]:
# Set MIN_FAILURES to 30 based on threshold sensitivity in 01_data_exploration.ipynb
MIN_FAILURES = 30  
eligible_summary = ld.drives_with_enough_failures(con, DATA_DIR, min_failures=MIN_FAILURES)
eligible_models = eligible_summary["model"].tolist()
eligible_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,model,drive_days,failures,failure_rate_pct
0,TOSHIBA MG07ACA14TA,3385931,133.0,0.0039
1,HGST HUH721212ALN604,911546,124.0,0.0136
2,ST8000NM0055,1212798,109.0,0.0090
3,ST12000NM0008,1714395,104.0,0.0061
4,TOSHIBA MG08ACA16TA,3616128,94.0,0.0026
5,HGST HUH721212ALE604,1197864,93.0,0.0078
6,WDC WUH722222ALE6L4,2870372,56.0,0.0020
7,ST8000DM002,814279,51.0,0.0063
8,ST16000NM001G,3029485,41.0,0.0014
9,ST14000NM001G,951836,37.0,0.0039


## 2. Which SMART columns are too sparse to use?

Computed with a single SQL pass (`AVG(CASE WHEN col IS NULL ...)` per column) rather
than loading all columns into pandas first.

In [3]:
null_fractions = cl.null_fraction_by_column(con, DATA_DIR)
null_fractions

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column,null_fraction
0,smart_255_raw,1.0000
1,smart_82_normalized,1.0000
2,smart_178_normalized,1.0000
3,smart_176_raw,1.0000
4,smart_15_normalized,1.0000
...,...,...
181,smart_9_raw,0.0014
182,smart_12_normalized,0.0014
183,smart_12_raw,0.0014
184,smart_194_normalized,0.0014


In [4]:
SPARSE_THRESHOLD = 0.05  # drop columns with >5% nulls; adjust after inspecting the real data's spread
sparse_cols = cl.sparse_columns(con, DATA_DIR, threshold=SPARSE_THRESHOLD)
print(f"Dropping {len(sparse_cols)} sparse columns:")
sparse_cols

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dropping 158 sparse columns:


['smart_255_raw',
 'smart_82_normalized',
 'smart_178_normalized',
 'smart_176_raw',
 'smart_15_normalized',
 'smart_15_raw',
 'smart_176_normalized',
 'smart_175_raw',
 'smart_175_normalized',
 'smart_164_raw',
 'smart_164_normalized',
 'smart_163_raw',
 'smart_163_normalized',
 'smart_161_raw',
 'smart_161_normalized',
 'smart_160_raw',
 'smart_160_normalized',
 'smart_178_raw',
 'smart_82_raw',
 'smart_251_normalized',
 'smart_255_normalized',
 'smart_211_normalized',
 'smart_211_raw',
 'smart_252_raw',
 'smart_252_normalized',
 'smart_212_raw',
 'smart_212_normalized',
 'smart_250_normalized',
 'smart_251_raw',
 'smart_250_raw',
 'smart_181_normalized',
 'smart_201_normalized',
 'smart_179_normalized',
 'smart_225_normalized',
 'smart_225_raw',
 'smart_182_raw',
 'smart_182_normalized',
 'smart_181_raw',
 'smart_179_raw',
 'smart_201_raw',
 'smart_245_normalized',
 'smart_13_normalized',
 'smart_13_raw',
 'smart_245_raw',
 'smart_11_raw',
 'smart_11_normalized',
 'smart_254_raw',
 

## 3. Raw vs. normalised: pick one per attribute

Backblaze reports both a raw and a normalised (vendor-scaled 1-253) value for many SMART
attributes. Keeping both may be redundant and may add multicollinearity to the statistical analysis.

The choice of using raw values simply comes from the idea that they are directly comparable across 
the quarter. In reliability literature, normalised scales may differ between manufacturers
and sectors, which could perplex any cross-model comparison in the real world. 

In [5]:
schema = ld.schema(con, DATA_DIR)
all_cols = list(schema["column_name"])

pairs = cl.raw_normalized_pairs(all_cols)
print(f"Found {len(pairs)} raw/normalized pairs:")
for raw_col, norm_col in pairs:
    print(f"  {raw_col}  <->  {norm_col}")

Found 93 raw/normalized pairs:
  smart_10_raw  <->  smart_10_normalized
  smart_11_raw  <->  smart_11_normalized
  smart_12_raw  <->  smart_12_normalized
  smart_13_raw  <->  smart_13_normalized
  smart_15_raw  <->  smart_15_normalized
  smart_160_raw  <->  smart_160_normalized
  smart_161_raw  <->  smart_161_normalized
  smart_163_raw  <->  smart_163_normalized
  smart_164_raw  <->  smart_164_normalized
  smart_165_raw  <->  smart_165_normalized
  smart_166_raw  <->  smart_166_normalized
  smart_167_raw  <->  smart_167_normalized
  smart_168_raw  <->  smart_168_normalized
  smart_169_raw  <->  smart_169_normalized
  smart_16_raw  <->  smart_16_normalized
  smart_170_raw  <->  smart_170_normalized
  smart_171_raw  <->  smart_171_normalized
  smart_172_raw  <->  smart_172_normalized
  smart_173_raw  <->  smart_173_normalized
  smart_174_raw  <->  smart_174_normalized
  smart_175_raw  <->  smart_175_normalized
  smart_176_raw  <->  smart_176_normalized
  smart_177_raw  <->  smart_177_nor

In [6]:
# Final feature list: smart_*_raw columns, excluding sparse ones
feature_cols = [c for c in all_cols if c.startswith("smart_") and c.endswith("_raw") and c not in sparse_cols]
print(f"{len(feature_cols)} feature columns selected:")
feature_cols

14 feature columns selected:


['smart_1_raw',
 'smart_3_raw',
 'smart_4_raw',
 'smart_5_raw',
 'smart_7_raw',
 'smart_9_raw',
 'smart_10_raw',
 'smart_12_raw',
 'smart_192_raw',
 'smart_193_raw',
 'smart_194_raw',
 'smart_197_raw',
 'smart_198_raw',
 'smart_199_raw']

## 4. Build the labeled snapshot dataset

One row per drive (restricted to eligible models),
with `label = 1` if the drive ever failed in the quarter and 0 otherwise, 
and SMART feature values taken from the day before failure (failed drives) 
or the last day in the dataset (healthy drives).

Implemented as a single SQL query using window functions to run directly against
the CSV folder, regardless of its size.

In [7]:
snapshot = cl.build_snapshot_dataset(
    con, DATA_DIR,
    eligible_models=eligible_models,
    feature_columns=feature_cols,
    out_path="../data/processed/snapshot_dataset.parquet"
)
print(snapshot.shape)
snapshot.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote ../data/processed/snapshot_dataset.parquet
(249058, 18)


,serial_number,model,snapshot_date,smart_1_raw,smart_3_raw,smart_4_raw,smart_5_raw,smart_7_raw,smart_9_raw,smart_10_raw,smart_12_raw,smart_192_raw,smart_193_raw,smart_194_raw,smart_197_raw,smart_198_raw,smart_199_raw,label
0,69J0A04SF97G,TOSHIBA MG07ACA14TA,2025-03-31,0,7891,9,0,0,45232,0,8,2,84,28,0,0,0,0
1,69J0A04WF97G,TOSHIBA MG07ACA14TA,2025-03-31,0,7719,22,0,0,45242,0,21,3,88,28,0,0,0,0
2,69J0A09FF97G,TOSHIBA MG07ACA14TA,2025-03-31,0,7864,18,0,0,45232,0,17,4,148,28,0,0,0,0
3,69L0A00LF97G,TOSHIBA MG07ACA14TA,2025-03-31,0,7965,21,0,0,45230,0,21,1,90,33,0,0,0,0
4,69L0A00RF97G,TOSHIBA MG07ACA14TA,2025-03-31,0,7769,9,0,0,45233,0,9,2,135,36,0,0,0,0


## 5. Class imbalance check

Shows that failure events are rare — therefore accuracy may not be a meaningful metric, 
First thought on analysis is to use rank-based tests (Mann-Whitney) for comparison, 
rather than assuming normally-distributed, balanced groups.

In [8]:
cl.class_balance_summary(snapshot)

,label,n,pct
0,0,248189,99.65
1,1,869,0.35


## 6. First-look EDA: do failed vs. healthy drives look different?

Quick distributional check per feature before statistical analysis.
Using median/IQR here rather than mean/std since these attributes may be typically
right-skewed with a long tail of high values for failing drives.

In [9]:
import pandas as pd

eda_rows = []
for col in feature_cols:
    healthy = snapshot.loc[snapshot["label"] == 0, col]
    failed = snapshot.loc[snapshot["label"] == 1, col]
    eda_rows.append({
        "feature": col,
        "healthy_median": healthy.median(),
        "failed_median": failed.median(),
        "healthy_p90": healthy.quantile(0.90),
        "failed_p90": failed.quantile(0.90),
    })

pd.DataFrame(eda_rows)

,feature,healthy_median,failed_median,healthy_p90,failed_p90
0,smart_1_raw,0.0,0.0,173769407.0,1.808710e+08
1,smart_3_raw,340.0,330.0,8018.0,7.975000e+03
2,smart_4_raw,10.0,13.0,21.0,2.900000e+01
3,smart_5_raw,0.0,9.0,0.0,1.406080e+04
4,smart_7_raw,0.0,0.0,880866977.1,2.165209e+09
5,smart_9_raw,27446.0,41391.0,53522.0,6.678540e+04
6,smart_10_raw,0.0,0.0,0.0,0.000000e+00
7,smart_12_raw,10.0,13.0,21.0,2.800000e+01
8,smart_192_raw,11.0,25.0,1448.0,1.941400e+03
9,smart_193_raw,896.0,1842.0,10976.0,7.533860e+04


**Reading this table:** a feature is a good candidate for statistical analysis if `failed_median` /
`failed_p90` are noticeably higher than the healthy equivalents. Attributes
`smart_4_raw`, `smart_5_raw`, `smart_9_raw`, `smart_12_raw`, `smart_192_raw`, `smart_193_raw`, and `smart_197_raw` 
degrade before failure, so they should show a visible gap, therefore qualify as good candidates. 
The rest of the attributes were not, so it's a useful negative control to confirm the method does not flag every single attribute.